<center><img src="https://keras.io/img/logo-small.png" alt="Keras logo" width="100"><br/>
Keras NLP — Disaster Tweets Classifier</center>

## 🌪️ NLP with Disaster Tweets — DistilBERT Fine-Tuning

Bu notebook **Kaggle NLP Getting Started** yarışması için hazırlanmıştır.
Gerçek felaket tweetlerini sahte olanlardan ayırt eden bir DistilBERT modeli eğitilmektedir.

### İyileştirmeler (v2)
- ✅ `keras_core` yerine modern `keras` + `keras_nlp`
- ✅ Tweet preprocessing (URL, mention, hashtag temizleme)
- ✅ `keyword` feature entegrasyonu
- ✅ EarlyStopping + ModelCheckpoint
- ✅ Learning Rate Scheduler
- ✅ Model Kaggle + HuggingFace export
- ✅ Detaylı metrik raporu (F1, Precision, Recall)

In [ ]:
# ─── Bağımlılıklar ───────────────────────────────────────────────────────────
!pip install -q keras-nlp==0.14.4 keras==3.3.3 tensorflow==2.16.1
!pip install -q tf-keras scikit-learn seaborn matplotlib

In [ ]:
# ─── Backend ayarı (JAX / TF / Torch) ────────────────────────────────────────
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import re
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_nlp
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"Keras NLP  : {keras_nlp.__version__}")
print(f"GPU mevcut : {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1️⃣ Veri Yükleme

In [ ]:
# Kaggle ortamı için path
TRAIN_PATH = "/kaggle/input/nlp-getting-started/train.csv"
TEST_PATH  = "/kaggle/input/nlp-getting-started/test.csv"
SAMPLE_SUB = "/kaggle/input/nlp-getting-started/sample_submission.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {df_train.shape}  |  Test : {df_test.shape}")
print(f"\nSınıf dağılımı:\n{df_train['target'].value_counts(normalize=True).round(3)}")
df_train.head(3)

## 2️⃣ Keşifsel Veri Analizi (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Tweet uzunluğu dağılımı
df_train['length'] = df_train['text'].str.len()
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    axes[0].hist(df_train[df_train['target']==label]['length'],
                 bins=30, alpha=0.6, color=color,
                 label='Not Disaster' if label==0 else 'Disaster')
axes[0].set_title('Tweet Uzunluğu Dağılımı')
axes[0].legend()

# Sınıf dengesi
counts = df_train['target'].value_counts()
axes[1].bar(['Not Disaster (0)', 'Disaster (1)'], counts.values,
            color=['steelblue', 'tomato'])
axes[1].set_title('Sınıf Dağılımı')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 30, str(v), ha='center')

# Boş keyword oranı
kw_null = df_train['keyword'].isna().mean() * 100
axes[2].bar(['Keyword Var', 'Keyword Yok'], [100-kw_null, kw_null],
            color=['mediumseagreen', 'salmon'])
axes[2].set_title('Keyword Doluluk Oranı (%)')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3️⃣ Metin Ön İşleme

In [ ]:
def clean_tweet(text: str) -> str:
    """Tweet metnini temizler ve normalleştirir."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\.\S+', '', text)      # URL
    text = re.sub(r'@\w+', '', text)                   # Mention
    text = re.sub(r'#(\w+)', r'\1', text)              # Hashtag (#flood → flood)
    text = re.sub(r'&amp;|&lt;|&gt;', ' ', text)       # HTML entity
    text = re.sub(r'[^\w\s!?.,;:\'"-]', ' ', text)    # Özel karakter
    text = re.sub(r'\s+', ' ', text).strip()           # Çoklu boşluk
    return text.lower()


def enrich_with_keyword(row) -> str:
    """Keyword varsa tweet başına ekler (ek bağlam sağlar)."""
    kw = row.get('keyword', '')
    text = row.get('clean_text', '')
    if pd.notna(kw) and kw:
        kw_clean = kw.replace('%20', ' ')
        return f"{kw_clean}: {text}"
    return text


# Uygula
for df in [df_train, df_test]:
    df['clean_text'] = df['text'].apply(clean_tweet)
    df['model_input'] = df.apply(enrich_with_keyword, axis=1)

# Örnek karşılaştırma
idx = 10
print("Orijinal :", df_train.loc[idx, 'text'])
print("Temizlenmiş:", df_train.loc[idx, 'model_input'])

## 4️⃣ Train / Validation Split

In [ ]:
# ─── Hiperparametreler ────────────────────────────────────────────────────────
PRESET         = "distil_bert_base_en_uncased"
SEQ_LEN        = 160
BATCH_SIZE     = 32
EPOCHS         = 6           # EarlyStopping ile en iyi nokta seçilir
LEARNING_RATE  = 2e-5
VAL_SPLIT      = 0.15
RANDOM_STATE   = 42
MODEL_SAVE_PATH = "distilbert_disaster"

X = df_train['model_input'].values
y = df_train['target'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, random_state=RANDOM_STATE, stratify=y
)

X_test = df_test['model_input'].values

print(f"Train : {X_train.shape[0]:,}  |  Val : {X_val.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

## 5️⃣ Model Kurulumu — DistilBERT Fine-Tuning

In [ ]:
keras.mixed_precision.set_global_policy('mixed_float16')  # hız + bellek

preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(
    PRESET,
    sequence_length=SEQ_LEN,
    name="distilbert_preprocessor"
)

classifier = keras_nlp.models.DistilBertClassifier.from_preset(
    PRESET,
    preprocessor=preprocessor,
    num_classes=2
)

classifier.summary(line_length=90)

In [ ]:
# ─── Callbacks ───────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.CSVLogger('training_log.csv')
]

# ─── Derleme ─────────────────────────────────────────────────────────────────
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    metrics=['accuracy']
)

print("Model derlendi ✓")

## 6️⃣ Eğitim

In [ ]:
history = classifier.fit(
    x=X_train,
    y=y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

## 7️⃣ Eğitim Eğrisi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(
    axes,
    ['accuracy', 'loss'],
    ['Accuracy', 'Loss']
):
    ax.plot(history.history[metric],       label='Train', marker='o')
    ax.plot(history.history[f'val_{metric}'], label='Val',   marker='s', linestyle='--')
    ax.set_title(f'Training {title}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8️⃣ Detaylı Değerlendirme

In [ ]:
def evaluate_split(X, y_true, split_name: str):
    """Bir split için tam metrik raporu üretir."""
    preds_raw = classifier.predict(X, batch_size=BATCH_SIZE, verbose=0)
    y_pred    = np.argmax(preds_raw, axis=1)

    print(f"\n{'='*50}")
    print(f"  {split_name.upper()} SET")
    print('='*50)
    print(classification_report(
        y_true, y_pred,
        target_names=['Not Disaster', 'Disaster'],
        digits=4
    ))

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=['Not Disaster', 'Disaster'],
        cmap='Blues', ax=ax
    )
    ax.set_title(f'Confusion Matrix — {split_name}')
    plt.tight_layout()
    plt.savefig(f'cm_{split_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return y_pred


y_val_pred = evaluate_split(X_val, y_val, "Validation")

## 9️⃣ Model Kaydetme (HuggingFace için)

In [ ]:
# Keras native format (.keras)
classifier.save(f"{MODEL_SAVE_PATH}.keras")
print(f"Model kaydedildi: {MODEL_SAVE_PATH}.keras")

# SavedModel formatı (TF Serving / ONNX dönüşümü için)
classifier.save(MODEL_SAVE_PATH)
print(f"SavedModel kaydedildi: {MODEL_SAVE_PATH}/")

# Model config ve ağırlıkları ayrı ayrı
import json
config = {
    'preset': PRESET,
    'seq_len': SEQ_LEN,
    'num_classes': 2,
    'labels': {0: 'Not Disaster', 1: 'Disaster'}
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("Config kaydedildi: model_config.json")

## 🔟 Submission Dosyası

In [ ]:
test_preds_raw = classifier.predict(X_test, batch_size=BATCH_SIZE)
test_preds     = np.argmax(test_preds_raw, axis=1)

sample_submission = pd.read_csv(SAMPLE_SUB)
sample_submission['target'] = test_preds
sample_submission.to_csv('submission.csv', index=False)

print(f"submission.csv hazır  |  Disaster tahmin oranı: {test_preds.mean():.2%}")
sample_submission.head()